In [1]:
import pandas as pd

# --------------------------------------------------
# 1. Read ABU 15-minute demand file
# --------------------------------------------------
abu = pd.read_csv(
    "demand_ABU_50K_kwh_15minIntervals.csv",
    sep=";",          # German-style separator
    decimal=","       # German decimal separator
)

# Check columns
print("ABU columns:", abu.columns.tolist())
abu.head()

ABU columns: ['Timestamp', 'Value']


,Timestamp,Value
0,01.01.2025 00:00,1.260790
1,01.01.2025 00:15,1.260790
2,01.01.2025 00:30,1.260790
3,01.01.2025 00:45,1.260790
4,01.01.2025 01:00,1.265566


In [2]:
# --------------------------------------------------
# 2. Parse timestamp and keep needed columns
# --------------------------------------------------
abu["Timestamp"] = pd.to_datetime(
    abu["Timestamp"],
    format="%d.%m.%Y %H:%M"
)

abu["Value"] = pd.to_numeric(abu["Value"], errors="coerce")

# Set timestamp as index
abu = abu.set_index("Timestamp")

In [3]:
# --------------------------------------------------
# 3. Convert to hourly demand
# --------------------------------------------------
abu_hourly = abu[["Value"]].resample("h").sum()
abu_hourly = abu_hourly.rename(columns={"Value": "abu_kwh"})

abu_hourly.head(20)

,abu_kwh
Timestamp,
2025-01-01 00:00:00,5.043162
2025-01-01 01:00:00,5.062264
2025-01-01 02:00:00,5.081367
2025-01-01 03:00:00,4.928544
2025-01-01 04:00:00,4.565589
2025-01-01 05:00:00,5.234190
2025-01-01 06:00:00,6.323055
2025-01-01 07:00:00,31.959126
2025-01-01 08:00:00,81.378288


In [8]:
ev = pd.read_excel(
    "EMobility_Demand.xlsx",
    skiprows=1
)

# Keep only the first two columns
ev = ev.iloc[:, :2].copy()

ev.columns = ["Timestamp", "ev_kwh"]

ev["Timestamp"] = pd.to_datetime(
    ev["Timestamp"] + "2025",
    format="%d.%m. %H:%M%Y",
    errors="coerce"
)

ev = ev.set_index("Timestamp")

In [9]:
# --------------------------------------------------
# 5. Combine both demands
# --------------------------------------------------
total_demand = abu_hourly.join(ev, how="left")


total_demand["total_kwh"] = total_demand["abu_kwh"] + total_demand["ev_kwh"]

# --------------------------------------------------
# 6. Make timestamp column = timesteps
# --------------------------------------------------
total_demand = total_demand.reset_index()
total_demand = total_demand.rename(columns={"Timestamp": "timesteps"})


In [12]:
total_demand.head(20)

,timesteps,abu_kwh,ev_kwh,total_kwh
0,2025-01-01 00:00:00,5.043162,0.00000,5.043162
1,2025-01-01 01:00:00,5.062264,0.00000,5.062264
2,2025-01-01 02:00:00,5.081367,0.00000,5.081367
3,2025-01-01 03:00:00,4.928544,0.00000,4.928544
4,2025-01-01 04:00:00,4.565589,0.00000,4.565589
5,2025-01-01 05:00:00,5.234190,0.00000,5.234190
6,2025-01-01 06:00:00,6.323055,0.00000,6.323055
7,2025-01-01 07:00:00,31.959126,0.00000,31.959126
8,2025-01-01 08:00:00,81.378288,0.00000,81.378288
9,2025-01-01 09:00:00,93.317591,6.89630,100.213891


In [13]:
final_hourly = total_demand[["timesteps", "total_kwh"]].copy()

final_hourly.head(20)

,timesteps,total_kwh
0,2025-01-01 00:00:00,5.043162
1,2025-01-01 01:00:00,5.062264
2,2025-01-01 02:00:00,5.081367
3,2025-01-01 03:00:00,4.928544
4,2025-01-01 04:00:00,4.565589
5,2025-01-01 05:00:00,5.234190
6,2025-01-01 06:00:00,6.323055
7,2025-01-01 07:00:00,31.959126
8,2025-01-01 08:00:00,81.378288
9,2025-01-01 09:00:00,100.213891


In [20]:
final_hourly["timesteps"] = pd.to_datetime(
    final_hourly["timesteps"]
).dt.strftime("%Y-%m-%d %H:%M:%S")

final_hourly.head()

,timesteps,total_kwh
0,2025-01-01 00:00:00,5.043162
1,2025-01-01 01:00:00,5.062264
2,2025-01-01 02:00:00,5.081367
3,2025-01-01 03:00:00,4.928544
4,2025-01-01 04:00:00,4.565589


In [22]:
final_hourly.to_csv("final_hourly.csv", index=False)